# TAMPanda + lego-coarse-simple

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import mujoco
from IPython.display import display as ipy_display
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})

from yaml_loader import load_task
from scenes import SimpleLegoScene
from bridges import LegoCoarseSimpleSLSBridge, LegoCoarseSimpleV2SLSBridge, execute_plan


In [ ]:
import os
if "DISPLAY" in os.environ:
    del os.environ["DISPLAY"]
os.environ["MUJOCO_GL"] = "egl"
os.environ["DISPLAY"] = ":3"

## Visualisation helpers
(same as in exercises)

In [ ]:
def _free_cam(env, lookat, distance, azimuth, elevation):
    cam = mujoco.MjvCamera(); mujoco.mjv_defaultFreeCamera(env.get_model(), cam)
    cam.lookat[:] = np.asarray(lookat, float)
    cam.distance, cam.azimuth, cam.elevation = distance, azimuth, elevation
    return cam

def _render(env, cam, width=640, height=480):
    env.forward(); r = mujoco.Renderer(env.get_model(), height=height, width=width)
    r.update_scene(env.get_data(), camera=cam); img = r.render(); r.close(); return img

def _show(img, title):
    fig, ax = plt.subplots(figsize=(5, 4)); ax.imshow(img); ax.axis("off")
    ax.set_title(title, fontsize=10); fig.tight_layout(); ipy_display(fig); plt.close(fig)

def ee_position(env):
    sid = mujoco.mj_name2id(env.get_model(), mujoco.mjtObj.mjOBJ_SITE, "attachment_site")
    env.forward(); return env.get_data().site_xpos[sid].copy()

def show_scene(env, title="Scene", distance=1.3, azimuth=135, elevation=-25, center=None):
    _show(_render(env, _free_cam(env, center if center is not None else _table_center, distance, azimuth, elevation)), title)

def show_top_view(env, center=None, distance=0.75, title="Top-down view"):
    _show(_render(env, _free_cam(env, center if center is not None else _table_center, distance, 90, -89)), title)

def show_ee_top_view(env, distance=0.35, title="End-effector (top-down)"):
    show_top_view(env, ee_position(env), distance, title)

def show_object_top_view(env, name, distance=0.22, title=None):
    show_top_view(env, env.get_object_position(name), distance, title or f"{name} (top-down)")

print("Visualisation helpers ready \u2713")


# Now with welds

In [ ]:
TASK = "dataset/ground_truth/tier4/task_001.yaml"
bricks, lat = load_task(TASK, TASK)
scene = SimpleLegoScene(bricks, seed=42)
scene.env.ik.solver = "daqp"
_table_center = (*scene.layout.asm_center, scene.table_z)   # used by the render helpers above
bricks

In [ ]:
from bridges import LegoCoarseSimpleV2SLSWeldBridge

# try new domain/bridge
bridge = LegoCoarseSimpleV2SLSWeldBridge.make_bridge("domains/lego_coarse_simple_v2.pddl", scene)
objects, goals = LegoCoarseSimpleV2SLSWeldBridge.build_objects_and_goal(bricks)
print("objects:", objects)
print("goals:  ", goals)

scene.to_target_structure()
show_scene(scene.env, "Target structure")
scene.env.reset()
scene.env.rest(1.0)

# problem = bridge.build_up_problem(objects, bridge.ground_state(objects), goals, "debug")
# print(problem)

show_scene(scene.env, "Initial scene")
show_top_view(scene.env, title="Initial scene (top-down)")
plan = bridge.plan(objects, goals)
print("\nPLAN:")
for step in plan:
    print(" ", step)

In [ ]:
scene.env.reset()
scene.env.rest(2.0)
viewer = scene.env.launch_viewer()

start_pos = scene.ee_pos()
ok, n = True, 0
for i, (name, args) in enumerate(plan):
    success, _ = bridge.execute_action(name, *args)
    print(f"[{i+1}/{len(plan)}] {name}({', '.join(args)}) -> {'ok' if success else 'FAILED'}")
    show_scene(scene.env, f"after {name}({', '.join(args)})")
    if not success:
        ok, n = False, i
        break
else:
    n = len(plan)

print("\nexecuted:", ok)

In [ ]:
scene.check_goal()

## Full Sweep of Tier 4:

In [22]:
import glob
SEED = 42

results = []

for i, task in enumerate(sorted(glob.glob("dataset/ground_truth/tier4/task_*.yaml"))):
    print(f"[{i+1}/100]: setting up {task}")
    bricks, lat = load_task(TASK, TASK)
    scene = SimpleLegoScene(bricks, seed=SEED)
    scene.env.ik.solver = "daqp"
    _table_center = (*scene.layout.asm_center, scene.table_z)   # used by the render helpers above

    
    bridge = LegoCoarseSimpleV2SLSWeldBridge.make_bridge("domains/lego_coarse_simple_v2.pddl", scene)
    objects, goals = LegoCoarseSimpleV2SLSWeldBridge.build_objects_and_goal(bricks)
    # print("objects:", objects)
    # print("goals:  ", goals)

    plan = bridge.plan(objects, goals)
    execute_plan(bridge, plan)
    success, _ = scene.check_goal()
    results.append(success)
    print(f"\nsuccess: {success}")
    print("\n\n")

[1/100]: setting up dataset/ground_truth/tier4/task_001.yaml
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 395 of `/u/prak10/lab/lego-workbenchmark/src/tampanda/tamp/domain_bridge.py`, you are using the following planning engine:
  * Engine name: pyperplan
  * Developers:  Albert-Ludwigs-Universität Freiburg (Yusra Alkhazraji, Matthias Frorath, Markus Grützner, Malte Helmert, Thomas Liebetraut, Robert Mattmüller, Manuela Ortlieb, Jendrik Seipp, Tobias Springenberg, Philip Stahl, Jan Wülfing)
  * Description: Pyperplan is a lightweight STRIPS planner written in Python.

[PickPlaceExecutor] Trying candidate 1/2: top_down_x
  EE error at grasp: 1.1 mm
  pick SUCCESS
[1/12] pick(b_4x2_brick_2) -> ok
[2/12] place(b_4x2_brick_2, l0, n2, n1) -> ok
[PickPlaceExecutor] Trying candidate 1/2: top_down_x


KeyboardInterrupt: 

In [ ]:
all(results)